# Notebook 01 : Synthetic Banking Data Generation
**Data sources:**
1. Ontario Top Baby Names CSV (data.ontario.ca) :  auto-downloaded
2. Faker en_CA :  addresses, phones, emails
3. src/document_generator.py :  7 banking document types

**Output:** `data/synthetic/documents.json` :  5,000 annotated docs

In [102]:
import os, json, random, requests
import pandas as pd
from faker import Faker
import sys; sys.path.append('..')
from src.document_generator import generate_dataset

fake = Faker('en_CA')
Faker.seed(42)
random.seed(42)

In [103]:
# Notebook 01 — New cell at top after imports
# Verify new generator loaded correctly

import importlib
import src.document_generator as dg
importlib.reload(dg)  # force reload in case old version cached

import inspect
print("File:", inspect.getfile(dg))
print("Generators:", len(dg.GENERATORS))
print("Names:", [g.__name__ for g in dg.GENERATORS])

# Quick single doc test — should complete in < 1 second
import time
start = time.time()
doc = dg.GENERATORS[0](names=["Priya Sharma", "Omar Farouq"])
print(f"\nSingle doc generated in {time.time()-start:.3f}s")
print(f"Doc type: {doc['doc_type']}")
print(f"Entities: {len(doc['entities'])}")
print(f"\nPreview:\n{doc['text'][:300]}...")


File: c:\MBA\Portfolio Projects\PII Redaction NER Pipeline\notebooks\..\src\document_generator.py
Generators: 7
Names: ['gen_kyc_narrative', 'gen_wire_transfer_narrative', 'gen_loan_narrative', 'gen_statement_narrative', 'gen_sar_narrative', 'gen_complaint_narrative', 'gen_internal_note_narrative']

Single doc generated in 0.002s
Doc type: KYC
Entities: 11

Preview:
Branch staff completed identity verification for Priya Sharma today. The customer confirmed their residential address as 9196 Jill Springs, Toronto, ON M3K 8Z0, postal code M3K 8Z0. Date of birth: 21/08/1987. SIN: 765-818-658. Best contact is johnsonjoshua@example.org or 780-450-4657. Account 10851-...


## Step 1 :  Download Ontario Baby Names (real open data)

In [104]:
MALE_URL   = 'https://data.ontario.ca/dataset/eb4c585c-6ada-4de7-8ff1-e876fb1a6b0b/resource/5d8b8ece-fa01-43c5-955b-4b642b28c559/download/baby_names_-_male.csv'
FEMALE_URL = 'https://data.ontario.ca/dataset/4d339626-98f9-49fe-aede-d64f03fa914f/resource/acc72e92-3100-4a04-8f5f-4fad9cd77cc5/download/baby_names_-_female_.csv'

os.makedirs('../data/raw', exist_ok=True)
for url, fname in [(MALE_URL,'ontario_names_male.csv'),(FEMALE_URL,'ontario_names_female.csv')]:
    p = f'../data/raw/{fname}'
    if not os.path.exists(p):
        r = requests.get(url, timeout=30)
        open(p,'wb').write(r.content)
        print(f'Downloaded {fname}')
    else:
        print(f'Already exists: {fname}')

Already exists: ontario_names_male.csv
Already exists: ontario_names_female.csv


In [105]:
male_df   = pd.read_csv('../data/raw/ontario_names_male.csv')
female_df = pd.read_csv('../data/raw/ontario_names_female.csv')
print('Columns:', male_df.columns.tolist())
print(male_df.head(3))

Columns: ['Year/Année', 'Name/Nom', 'Frequency/Fréquence']
   Year/Année Name/Nom  Frequency/Fréquence
0        1917    AARON                    7
1        1917      ABE                    5
2        1917     ABIE                    5


In [106]:
male_df.columns

Index(['Year/Année', 'Name/Nom', 'Frequency/Fréquence'], dtype='object')

In [107]:
print('Columns:', female_df.columns.tolist())
print(female_df.head(3))

Columns: ['Year/Année', 'Name/Nom', 'Frequency/Fréquence']
   Year/Année  Name/Nom  Frequency/Fréquence
0        1913  MARGARET                    6
1        1913      MARY                    7
2        1914    GLADYS                    6


In [108]:
# Adapt column name if needed

col = 'Name/Nom' if 'Name/Nom' in male_df.columns else male_df.columns[0]
first_names = list(set(
    male_df[col].dropna().str.title().tolist() +
    female_df[col].dropna().str.title().tolist()
))
# Combine with Faker last names
# NEW — 10,000 names, split into separate train/test pools
random.seed(42)

# Generate 10,000 unique full names
# Each first name gets 5 different last names
all_names = []
for fn in first_names:
    for _ in range(5):
        all_names.append(f"{fn} {fake.last_name()}")

# Shuffle and deduplicate
random.shuffle(all_names)
all_names = list(dict.fromkeys(all_names))[:10000]

# CRITICAL — separate pools so test sees unseen names
TRAIN_NAMES = all_names[:8000]   # model learns from these
TEST_NAMES  = all_names[8000:]   # model never sees these

print(f"Train name pool: {len(TRAIN_NAMES)}")
print(f"Test name pool:  {len(TEST_NAMES)}")
print(f"Sample train: {TRAIN_NAMES[:3]}")
print(f"Sample test:  {TEST_NAMES[:3]}")

Train name pool: 8000
Test name pool:  2000
Sample train: ['Alonso Carpenter', 'Makenzie                       Graves', 'Dhriti Munoz']
Sample test:  ['Bane Spence', 'Samantha Walker', 'Mégane Adams']


## Step 2 :  Generate 5,000 Documents

In [109]:
# Generate 6,000 train docs with TRAIN_NAMES
# Generate 1,000 test docs with TEST_NAMES

train_docs = generate_dataset(n=128, names=TRAIN_NAMES)
test_docs  = generate_dataset(n=22, names=TEST_NAMES)

# Tag each with split so Notebook 02 knows which pool it came from
for d in train_docs: d['split'] = 'train'
for d in test_docs:  d['split'] = 'test'

all_docs = train_docs + test_docs
random.shuffle(all_docs)

import os, json
os.makedirs('../data/synthetic', exist_ok=True)
with open('../data/synthetic/documents.json', 'w') as f:
    json.dump(all_docs, f, indent=2)

print(f"Total docs: {len(all_docs)}")
print(f"Train docs: {len(train_docs)}")
print(f"Test docs:  {len(test_docs)}")



#   1. KYC Form                 (structured form)
#   2. Wire Transfer Request    (semi-structured)
#   3. Personal Loan Application(structured form)
#   4. Account Statement Header (structured)
#   5. SAR Memo                 (structured + narrative paragraph)
#   6. Customer Complaint Letter(mostly unstructured prose)
#   7. Internal Bank Note       (pure freeform narrative — most realistic)
from collections import Counter
print('By type:', Counter(d['doc_type'] for d in all_docs))

Total docs: 150
Train docs: 128
Test docs:  22
By type: Counter({'KYC': 23, 'WIRE_TRANSFER': 22, 'SAR_MEMO': 21, 'INTERNAL_NOTE': 21, 'ACCOUNT_STATEMENT': 21, 'LOAN_APPLICATION': 21, 'COMPLAINT_LETTER': 21})


In [115]:
# Preview
s = all_docs[3]
print('=== TYPE:', s['doc_type'], '===')
print(s['text'])
print('ENTITIES:')
for st,en,lbl in s['entities']:
    print(f'  [{lbl}] {s["text"][st:en]!r}')

=== TYPE: WIRE_TRANSFER ===
Transfer authorization MHL-86501976: Aurelio Martin has requested that CAD 90,813.00 be transferred from account 74408-010-1239473 at Meridian Credit Union to Mathilde Foster. Destination bank is National Bank of Canada, account 25338-004-7100567, SWIFT AKCCCAYT. Contact the sender at 604.654.8409 or dawnwalker@example.org for queries. Beneficiary contact: fflores@example.org. Postal reference: M3Z 7G6. The institution has fulfilled its due diligence obligations. This document is governed by the Bank Act (R.S.C., 1991, c. 46).
ENTITIES:
  [PERSON] 'Aurelio Martin'
  [PERSON] 'Mathilde Foster'
  [EMAIL] 'dawnwalker@example.org'
  [EMAIL] 'fflores@example.org'
  [PHONE] '604.654.8409'
  [ACCOUNT_NO] '74408-010-1239473'
  [ACCOUNT_NO] '25338-004-7100567'
  [ORG] 'Meridian Credit Union'
  [ORG] 'National Bank of Canada'
  [SWIFT] 'AKCCCAYT'
  [AMOUNT] 'CAD 90,813.00'
  [POSTAL_CODE] 'M3Z 7G6'


In [111]:
# Entity distribution
all_labels = [lbl for d in all_docs for _,_,lbl in d['entities']]
print('Entity counts:')
for lbl,cnt in Counter(all_labels).most_common():
    print(f'  {lbl:<20} {cnt}')

Entity counts:
  PERSON               214
  ORG                  193
  ACCOUNT_NO           172
  AMOUNT               169
  EMAIL                168
  POSTAL_CODE          164
  PHONE                150
  ADDRESS              128
  SIN                  86
  CREDIT_CARD          61
  DOB                  60
  TRANSIT_NO           23
  SWIFT                22


In [112]:
# NEW — also show split breakdown
from collections import Counter
type_counts = Counter(d['doc_type'] for d in all_docs)
print('Documents by type:')
for k, v in type_counts.most_common():
    print(f'  {k:<25} {v}')

print('\nBy split:')
print(f"  train pool: {sum(1 for d in all_docs if d['split']=='train')}")
print(f"  test pool:  {sum(1 for d in all_docs if d['split']=='test')}")

Documents by type:
  KYC                       23
  WIRE_TRANSFER             22
  SAR_MEMO                  21
  INTERNAL_NOTE             21
  ACCOUNT_STATEMENT         21
  LOAN_APPLICATION          21
  COMPLAINT_LETTER          21

By split:
  train pool: 128
  test pool:  22


In [113]:
import os, json

# Recreate directory if it doesn't exist
os.makedirs('../data/synthetic', exist_ok=True)

# Save
with open('../data/synthetic/documents.json', 'w') as f:
    json.dump(all_docs, f, indent=2)

# Verify
size = os.path.getsize('../data/synthetic/documents.json') / (1024*1024)
print(f'✅ Saved {len(all_docs)} documents ({size:.1f} MB)')
print(f'   Train pool: {sum(1 for d in all_docs if d["split"]=="train")}')
print(f'   Test pool:  {sum(1 for d in all_docs if d["split"]=="test")}')

✅ Saved 150 documents (0.2 MB)
   Train pool: 128
   Test pool:  22


## Done :  Next: 02_annotation_prep.ipynb

In [ ]:
# import subprocess
# result = subprocess.run(['pip', 'freeze'], capture_output=True, text=True)

# packages = [
#     'spacy', 'thinc', 'numpy', 'faker', 'pandas',
#     'requests', 'streamlit', 'scikit-learn', 'matplotlib',
#     'seaborn', 'huggingface-hub'
# ]

# print("# Paste this into requirements.txt")
# print()
# for line in result.stdout.split('\n'):
#     if any(p.lower() in line.lower() for p in packages):
#         print(line)

# Paste this into requirements.txt

en-core-web-trf @ https://github.com/explosion/spacy-models/releases/download/en_core_web_trf-3.7.3/en_core_web_trf-3.7.3-py3-none-any.whl#sha256=f72abb34bdf174876bd4267b29b2501677e605e0a251fdc56c163003182ed68b
Faker==40.11.0
matplotlib==3.10.8
matplotlib-inline==0.1.7
numpy==1.26.4
opentelemetry-instrumentation-requests==0.52b1
pandas==2.3.3
requests==2.32.3
requests-oauthlib==2.0.0
scikit-learn==1.8.0
seaborn==0.13.2
spacy==3.7.5
spacy-curated-transformers==0.2.2
spacy-legacy==3.0.12
spacy-loggers==1.0.5
streamlit==1.55.0
thinc==8.2.5


Here's the full breakdown of what's in Version 4:

What Changed
Fillers — 52 total across 4 pools:
FILLER_GENERAL      → 20 sentences  (processing, confidentiality)
FILLER_LEGAL        → 12 sentences  (Bank Act, PIPEDA, FINTRAC)
FILLER_INTERNAL     → 12 sentences  (escalation, review, audit)
FILLER_PROCEDURAL   → 10 sentences  (verification, notifications)
8 templates per generator (56 total):
KYC           → 8 templates (formal, onboarding, regulatory, branch note...)
Wire Transfer → 8 templates (memo, authorization, instruction, branch...)
Loan          → 8 templates (application, assessment, consultation...)
Statement     → 8 templates (monthly, summary, letter, billing...)
SAR           → 8 templates (report, analyst flag, compliance, AML...)
Complaint     → 8 templates (formal, informal, terse, bullet style...)
Internal Note → 8 templates (handoff, branch note, file update, memo...)
More randomisation:
ca_phone()   → 4 phone formats (+1 xxx, (xxx), xxx-xxx, xxx.xxx)
ca_dob()     → 4 date formats (ISO, DD/MM/YYYY, July 22 1984, 22 July 1984)
dollars()    → 4 amount formats ($42,500, CAD 42,500, $42,500 CAD, CAD$42,500)
ref_no()     → random reference numbers added to most docs
More cities  → 21 cities across 6 provinces (was 14)
More banks   → 15 banks including ATB, Meridian
More area codes → 19 Canadian area codes